In [5]:
import pandas as pd
import numpy as np
import sqlite3
# Conectamos con la base de datos cervezas.db
connection = sqlite3.connect('eventos.db')

# Obtenemos un cursor que utilizaremos para hacer las queries
crsr = connection.cursor()
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)


In [6]:
# Borramos tabla para pruebas
query = f'drop table if exists eventos'
crsr.execute(query);
# query = f'drop table if exists users'
# crsr.execute(query);
# query = f'drop table if exists empleados'
# crsr.execute(query);
# query = f'drop table if exists reparto'
# crsr.execute(query);
# query = f'drop table if exists reparto'
# crsr.execute(query);
# query = f'drop table if exists reparto'
# crsr.execute(query);

In [8]:
# Crear tabla eventos
sql_cre = f'''
CREATE TABLE eventos (
    id INT AUTO_INCREMENT PRIMARY KEY,

    business_id VARCHAR(50),
    fuente VARCHAR(255),
    external_id VARCHAR(255),

    title VARCHAR(255),
    description TEXT,

    categoria VARCHAR(100),
    tipo_plantilla VARCHAR(100),

    municipio VARCHAR(100),
    territorio VARCHAR(50),
    address VARCHAR(255),

    lat DECIMAL(10,7),
    lng DECIMAL(10,7),

    telefono VARCHAR(50),
    email VARCHAR(255),
    website VARCHAR(255),

    es_interior BOOLEAN,
    es_carrito BOOLEAN,
    es_cambiador BOOLEAN,
    es_silla_ruedas BOOLEAN,
    es_mascotas BOOLEAN,

    edad_minima INT,

    fecha_inicio DATE,
    fecha_fin DATE,

    lugar VARCHAR(255),
    price VARCHAR(50),
    imagen_url TEXT,
    tipo_evento VARCHAR(100)
);
'''

crsr.execute(sql_cre)

In [10]:
# 1. Leer CSV
df = pd.read_csv("events_2026-06-04.csv", sep=";")

# 2. Reemplazar NaN por None (para que SQL los acepte como NULL)
df = df.where(pd.notnull(df), None)

# 3. Preparar columnas y placeholders
columnas = list(df.columns)
cols_sql = ", ".join(columnas)
placeholders = ", ".join(["?"] * len(columnas))   # Si usas SQLite
# Para MySQL sería: "%s" en vez de "?"

sql_ins = f"INSERT INTO eventos ({cols_sql}) VALUES ({placeholders})"

# 4. Insertar fila a fila
for _, fila in df.iterrows():
    valores = tuple(fila.values)
    crsr.execute(sql_ins, valores)

# crsr.commit()
print("Inserción completada")

Inserción completada


In [11]:
connection.commit() # Commit por si nos queda algo pendiente antes de cerrar conexión
connection.close()